# Day 10: End-to-End Orchestration, Observability & Verification
This notebook provides a complete E2E walkthrough of a synthetic patient from raw JSON to Gold metric.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_date, date_add, udf, to_date
from pyspark.sql.types import StringType
import hashlib

spark = SparkSession.builder.appName("Day10_E2E").getOrCreate()

print("=== EHDIP E2E SIMULATION ===\n")

# 1. RAW INGESTION (Bronze)
raw_data = [
    ("P-100", "123-45-6789", "2023-01-01", 9201, "2023-01-01", "2023-01-05"), # Index Admission
    ("P-100", "123-45-6789", "2023-01-20", 9201, "2023-01-20", "2023-01-22")  # Readmission (within 30 days)
]
cols = ["patient_id", "ssn", "dob", "condition_concept_id", "start_date", "end_date"]
df_raw = spark.createDataFrame(raw_data, cols)
df_raw = df_raw.withColumn("start_date", to_date(col("start_date"))).withColumn("end_date", to_date(col("end_date")))
print("1. RAW DATA INGESTED:")
df_raw.show()

# 2. TOKENIZATION / DE-ID
def hash_ssn(val):
    import os
    salt = os.environ.get('EHDIP_HASH_SALT', '')
    return hashlib.sha256((val + salt).encode('utf-8')).hexdigest()[:10] if val else None
hash_udf = udf(hash_ssn, StringType())

df_deid = df_raw.withColumn("ssn_token", hash_udf(col("ssn"))).drop("ssn")
print("2. DATA AFTER TOKENIZATION:")
df_deid.show()

# 3. OMOP SILVER STANDARDIZATION
df_silver = df_deid.select(
    col("patient_id").alias("person_id"),
    col("condition_concept_id"),
    col("start_date").alias("condition_start_date"),
    col("end_date").alias("condition_end_date")
)
print("3. OMOP SILVER FORMAT (condition_occurrence):")
df_silver.show()

# 4. DATA QUALITY (Mock check: start_date <= end_date)
df_dq = df_silver.withColumn("is_valid", col("condition_start_date") <= col("condition_end_date"))
df_valid = df_dq.filter(col("is_valid")).drop("is_valid")
print("4. DATA PASSED DQ CHECKS:")
df_valid.show()

# 5. GOLD METRIC: 30-Day Readmission
df_valid.createOrReplaceTempView("silver_conditions")
gold_query = """
WITH admissions AS (
    SELECT person_id, condition_start_date as admission_date, condition_end_date as discharge_date
    FROM silver_conditions
    WHERE condition_concept_id = 9201
)
SELECT 
    a1.person_id,
    a1.discharge_date,
    a2.admission_date as readmission_date,
    CASE WHEN a2.admission_date IS NOT NULL THEN 1 ELSE 0 END as is_30d_readmission
FROM admissions a1
LEFT JOIN admissions a2 
    ON a1.person_id = a2.person_id 
    AND a2.admission_date > a1.discharge_date 
    AND a2.admission_date <= date_add(a1.discharge_date, 30)
WHERE a1.admission_date = '2023-01-01' -- Focus on index admission
"""
df_gold = spark.sql(gold_query)
print("5. GOLD MART (Readmission Metric):")
df_gold.show()
print("E2E Pipeline Walkthrough Complete.")